# Q1 – TF-IDF from Scratch

Using ShopSense Reviews Dataset. No sklearn for the core TF-IDF computation.

In [ ]:
import re
import math
import numpy as np
import pandas as pd
from collections import Counter
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

REVIEWS_PATH = "shopsense_reviews-2.csv"
QUERY        = "wireless earbuds battery life poor"


In [ ]:
def load_reviews(path):
    try:
        df = pd.read_csv(path)
        assert "review_text" in df.columns and "category" in df.columns
        df["review_text"] = df["review_text"].fillna("").astype(str)
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {path}")

df = load_reviews(REVIEWS_PATH)
print("Shape:", df.shape)
print("Categories:", df["category"].value_counts().to_dict())


In [ ]:
def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text.lower())
    return text.split()

def build_vocabulary(docs):
    vocab = {}
    for tokens in docs:
        for t in tokens:
            if t not in vocab:
                vocab[t] = len(vocab)
    return vocab

tokenized = [clean_text(t) for t in df["review_text"]]
vocab     = build_vocabulary(tokenized)
print("Vocab size:", len(vocab))


In [ ]:
def compute_tf(tokens):
    n = len(tokens)
    if n == 0:
        return {}
    counts = Counter(tokens)
    return {w: c / n for w, c in counts.items()}

def compute_idf(tokenized_docs, vocab):
    N   = len(tokenized_docs)
    df_ = np.zeros(len(vocab), dtype=np.float64)
    for tokens in tokenized_docs:
        for w in set(tokens):
            if w in vocab:
                df_[vocab[w]] += 1
    idf = np.log((N + 1) / (df_ + 1)) + 1
    return idf

idf_vector = compute_idf(tokenized, vocab)
print("IDF computed. Sample (first 5):", idf_vector[:5])


In [ ]:
def build_tfidf_sparse(tokenized_docs, vocab, idf_vector):
    rows, cols, vals = [], [], []
    for doc_idx, tokens in enumerate(tokenized_docs):
        tf = compute_tf(tokens)
        for w, tf_val in tf.items():
            if w in vocab:
                col = vocab[w]
                rows.append(doc_idx)
                cols.append(col)
                vals.append(tf_val * idf_vector[col])
    mat = csr_matrix((vals, (rows, cols)), shape=(len(tokenized_docs), len(vocab)))
    norms = np.sqrt(mat.power(2).sum(axis=1))
    norms[norms == 0] = 1
    mat = mat.multiply(1 / norms)
    return mat

tfidf_matrix = build_tfidf_sparse(tokenized, vocab, idf_vector)
print("TF-IDF matrix shape:", tfidf_matrix.shape)


## (b) Query: Top-5 most relevant reviews via cosine similarity

In [ ]:
def rank_query(query_text, tfidf_matrix, idf_vector, vocab, df, top_n=5):
    q_tokens = clean_text(query_text)
    q_tf     = compute_tf(q_tokens)
    q_vec    = np.zeros(len(vocab))
    for w, tf_val in q_tf.items():
        if w in vocab:
            q_vec[vocab[w]] = tf_val * idf_vector[vocab[w]]
    norm = np.linalg.norm(q_vec)
    if norm > 0:
        q_vec /= norm
    scores      = tfidf_matrix.dot(q_vec)
    top_indices = np.argsort(scores)[::-1][:top_n]
    results = df.iloc[top_indices][["review_id", "category", "review_text", "rating"]].copy()
    results["cosine_score"] = scores[top_indices]
    return results

top5 = rank_query(QUERY, tfidf_matrix, idf_vector, vocab, df)
print(top5[["review_id", "category", "cosine_score", "review_text"]].to_string(index=False))


## (c) Compare with sklearn TfidfVectorizer – average L2 difference

In [ ]:
def compare_with_sklearn(df, vocab, tfidf_matrix):
    sk_vec = TfidfVectorizer(
        preprocessor=lambda x: re.sub(r"<[^>]+>", " ", x),
        token_pattern=r"[a-zA-Z]+",
        lowercase=True,
        smooth_idf=True,
        norm="l2"
    )
    sk_matrix = sk_vec.fit_transform(df["review_text"])

    shared_words  = set(vocab.keys()) & set(sk_vec.vocabulary_.keys())
    scratch_cols  = [vocab[w] for w in shared_words]
    sklearn_cols  = [sk_vec.vocabulary_[w] for w in shared_words]

    scratch_sub = tfidf_matrix[:, scratch_cols].toarray()
    sklearn_sub = sk_matrix[:, sklearn_cols].toarray()

    avg_l2 = np.mean(np.linalg.norm(scratch_sub - sklearn_sub, axis=1))
    print(f"Shared vocab size: {len(shared_words)}")
    print(f"Average L2 difference (scratch vs sklearn): {avg_l2:.6f}")
    return avg_l2

avg_l2 = compare_with_sklearn(df, vocab, tfidf_matrix)


## (d) Highest avg TF-IDF word in Electronics category

In [ ]:
def top_word_in_category(df, tfidf_matrix, vocab, category="Electronics"):
    mask   = (df["category"] == category).values
    subset = tfidf_matrix[mask]
    avg_scores = np.asarray(subset.mean(axis=0)).flatten()
    top_idx  = int(np.argmax(avg_scores))
    idx_to_word = {v: k for k, v in vocab.items()}
    top_word  = idx_to_word[top_idx]
    top_score = avg_scores[top_idx]
    print(f"Category: {category}")
    print(f"Top word: '{top_word}' | Avg TF-IDF: {top_score:.6f}")
    return top_word, top_score

top_word, top_score = top_word_in_category(df, tfidf_matrix, vocab, "Electronics")


## Explanation for (d)

The top word in the Electronics category is the one that appears frequently within Electronics reviews but rarely in other categories or across the full 10K corpus.
This means it has both a high TF (used often in Electronics docs) and a high IDF (not very common globally), so their product is maximized.
Words like generic stop words would have near-zero IDF and would not rank at the top, while very rare words would have high IDF but low TF.
The highest-ranking word sits at the sweet spot where it is common enough in Electronics to have notable TF but specific enough to carry a meaningful IDF penalty.
